# 🚀 FaceFlux — 100% Free Google Colab GPU Backend
### NVIDIA T4 GPU (15GB VRAM) for Super-Fast Video Generation

**Kaise use karein:**
1. Menu me jaao: `Runtime` -> `Change runtime type` -> select **T4 GPU**.
2. Niche diye gaye cells ko **Play (Run)** button dabakar run karo.
3. Cell 2 ke output mein aakhri me ek **Public Tunnel URL** milega (jaise `https://xxxx.loca.lt`).
4. Wo URL FaceFlux ke `.env` file mein `COLAB_GPU_URL=...` ke roop mein save kar lo!

In [ ]:
# Step 1: Install required packages on Google Colab
!pip install -q fastapi uvicorn diffusers transformers accelerate imageio[ffmpeg]
!npm install -g localtunnel

In [ ]:
# Step 2: Start FaceFlux Colab Video Engine
import io
import time
import base64
import tempfile
import threading
from pathlib import Path
from PIL import Image
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import torch
from diffusers import AnimateDiffPipeline, MotionAdapter, EulerDiscreteScheduler
from diffusers.utils import export_to_video

app = FastAPI(title="FaceFlux Colab Engine")

print("Loading AI Video Generation Model onto Colab GPU...")
device = "cuda" if torch.cuda.is_available() else "cpu"
adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2", torch_dtype=torch.float16)
pipe = AnimateDiffPipeline.from_pretrained(
    "SG161222/Realistic_Vision_V5.1_noVAE",
    motion_adapter=adapter,
    torch_dtype=torch.float16
)
scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing="trailing", beta_schedule="linear")
pipe.scheduler = scheduler

# Enable VAE optimizations safely across diffusers versions
if hasattr(pipe, "vae") and hasattr(pipe.vae, "enable_slicing"):
    pipe.vae.enable_slicing()
if hasattr(pipe, "vae") and hasattr(pipe.vae, "enable_tiling"):
    pipe.vae.enable_tiling()

pipe.to(device)
if torch.cuda.is_available():
    print("✓ Model loaded successfully on GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Running on CPU (Make sure to select T4 GPU in Runtime settings)")

class GenRequest(BaseModel):
    image: str
    prompt: str
    duration: int = 5

@app.post("/generate")
async def generate(req: GenRequest):
    img_bytes = base64.b64decode(req.image)
    init_img = Image.open(io.BytesIO(img_bytes)).convert("RGB").resize((512, 512))
    
    frames = max(16, min(req.duration * 8, 32))
    print(f"Rendering video for prompt: '{req.prompt}' ({frames} frames)...")
    
    output = pipe(
        prompt=req.prompt,
        negative_prompt="bad quality, distorted, watermark, deformed, ugly",
        num_frames=frames,
        guidance_scale=7.5,
        num_inference_steps=25,
        generator=torch.manual_seed(42),
    ).frames[0]
    
    temp_mp4 = Path(tempfile.gettempdir()) / "colab_output.mp4"
    export_to_video(output, str(temp_mp4), fps=8)
    
    with open(temp_mp4, "rb") as f:
        b64_video = base64.b64encode(f.read()).decode("utf-8")
        
    return {"status": "COMPLETED", "video": b64_video}

# Start server in thread to avoid Jupyter event loop conflict
def start_server():
    config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
    server = uvicorn.Server(config)
    server.run()

threading.Thread(target=start_server, daemon=True).start()
time.sleep(2)
print("✓ FaceFlux Server started on localhost:8000!")
print("Starting tunnel to generate public URL...")

# Expose via localtunnel
!npx localtunnel --port 8000